## Wprowadzenie do zaawansowanego systemu typów (*Playing With Python Types, Part 1*)

Do tej pory w adnotacjach typów (*type hints*) używaliśmy wyłącznie podstawowych typów, takich jak `str`, `float` oraz `bool`. System typów w Pythonie jest jednak znacznie bardziej rozbudowany i obsługuje wiele rodzajów bardziej złożonych struktur. Jest to konieczne, ponieważ musi on być w stanie odpowiednio odwzorować dynamiczną naturę Pythona opartą na **duck typingu**.

W tej części poznamy bliżej ten system typów, implementując prostą grę karcianą. Zobaczymy, jak określać:
* typy **sekwencji oraz mapowań**, takich jak krotki (`tuple`), listy (`list`) i słowniki (`dict`),
* **aliasy typów** (*type aliases*), które ułatwiają czytanie kodu,
* sytuacje, w których funkcje i metody **niczego nie zwracają** (`None`),
* obiekty, które mogą być **dowolnego typu** (`Any`).

> Po krótkiej dygresji na temat **teorii typów** przyjrzymy się również kolejnym sposobom określania typów w Pythonie.

### Poniższy przykład pokazuje implementację zwykłej (francuskiej) talii kart:

In [18]:
import random

SUITS = "♠ ♡ ♢ ♣".split()
RANKS = "2 3 4 5 6 7 8 9 10 J Q K A".split()

def create_deck(shuffle=False):
    """Create a new deck of 52 cards"""
    deck = [(s, r) for r in RANKS for s in SUITS]
    if shuffle:
        random.shuffle(deck)
    return deck

def deal_hands(deck):
    """Deal the cards in the deck into four hands"""
    return (deck[0::4], deck[1::4], deck[2::4], deck[3::4])

def play():
    """Play a 4-player card game"""
    deck = create_deck(shuffle=True)
    names = "P1 P2 P3 P4".split()
    hands = {n: h for n, h in zip(names, deal_hands(deck))}

    for name, cards in hands.items():
        card_str = " ".join(f"{s}{r}" for (s, r) in cards)
        print(f"{name}: {card_str}")

if __name__ == "__main__":
    play()

P1: ♠6 ♠2 ♣6 ♣2 ♠3 ♢A ♢J ♣8 ♡9 ♡10 ♣7 ♡7 ♠7
P2: ♢K ♣Q ♡Q ♣J ♢5 ♢2 ♢3 ♠5 ♣K ♢9 ♣9 ♣3 ♡4
P3: ♢8 ♠9 ♡5 ♣5 ♠8 ♣10 ♡J ♡K ♠Q ♣4 ♢7 ♠J ♠K
P4: ♠A ♡A ♢10 ♣A ♠4 ♡6 ♠10 ♡8 ♡2 ♢4 ♡3 ♢6 ♢Q


Każda karta jest reprezentowana jako krotka (`tuple`) łańcuchów dwóch znaków (`str`) oznaczających kolor (*suit*) oraz figurę/wartość (*rank*). Talia jest reprezentowana jako lista (`list`) kart. Funkcja **`create_deck()`** tworzy standardową talię 52 kart do gry i opcjonalnie ją tasuje. Funkcja **`deal_hands()`** rozdaje talię kart czterem graczom.

Na koniec funkcja **`play()`** uruchamia rozgrywkę. Na ten moment jedynie przygotowuje ona grę karcianą poprzez utworzenie potasowanej talii i rozdanie kart każdemu z graczy.

### Sekwencje i mapowania (*Sequences and Mappings*)

Dodajmy teraz adnotacje typów do naszej gry karcianej — innymi słowy, otypujmy funkcje `create_deck()`, `deal_hands()` oraz `play()`. Pierwszym wyzwaniem jest konieczność opisania typów złożonych, takich jak lista reprezentująca talię kart oraz krotki reprezentujące poszczególne karty.

W przypadku prostych typów, takich jak `str`, `float` czy `bool`, dodawanie adnotacji sprowadza się po prostu do użycia nazwy samego typu:

In [8]:
name: str = "Guido"
pi: float = 3.142
centered: bool = False
type(name)
type(pi)
type(centered)

bool

To samo można zrobić ze złożonymi typami:

In [14]:
names: list = ["Guido", "Jukka", "Ivan"]
version: tuple = (3, 7, 1)
options: dict = {"centered": False, "capitalize": True}
type(names[2])
type(version[0])
type(options["centered"])

bool

Nie oddaje to jednak pełnego obrazu sytuacji. Jakiego typu będą wyrażenia `names[2]`, `version[0]` oraz `options["centered"]`? W tym konkretnym przypadku widać, że są to odpowiednio `str`, `int` oraz `bool`, jednak same podstawowe adnotacje typów nie dostarczają na ten temat żadnej informacji.

Zamiast tego należy użyć specjalnych typów zdefiniowanych w module `typing`. Wprowadzają one składnię pozwalającą określić typy elementów wewnątrz typów złożonych. Możemy to zapisać w następujący sposób:

In [15]:
from typing import Dict, List, Tuple

names: List[str] = ["Guido", "Jukka", "Ivan"]
version: Tuple[int, int, int] = (3, 7, 1)
options: Dict[str, bool] = {"centered": False, "capitalize": True}

Zwróćmy uwagę, że każdy z tych typów zaczyna się wielką literą i wykorzystuje nawiasy kwadratowe do określenia typów poszczególnych elementów: `names` jest listą napisów, `version` to trójelementowa krotka składająca się z trzech liczb całkowitych, natomiast `options` jest słownikiem mapującym napisy na wartości logiczne. Moduł `typing` zawiera znacznie więcej typów złożonych, takich jak `Counter`, `Deque`, `FrozenSet`, `NamedTuple` oraz `Set`, a także inne rodzaje typów.

Wróćmy teraz do gry karcianej. Pojedyncza karta jest reprezentowana przez krotkę dwóch napisów, co możemy zapisać jako `Tuple[str, str]`, dzięki czemu typ całej talii kart przyjmuje postać `List[Tuple[str, str]]`. W związku z tym funkcję `create_deck()` możemy opatrzyć adnotacjami w następujący sposób:

In [22]:
def create_deck(shuffle: bool = False) -> List[Tuple[str, str]]:
    """Create a new deck of 52 cards"""
    deck = [(s, r) for r in RANKS for s in SUITS]
    if shuffle:
        random.shuffle(deck)
    return deck
create_deck()

[('♠', '2'),
 ('♡', '2'),
 ('♢', '2'),
 ('♣', '2'),
 ('♠', '3'),
 ('♡', '3'),
 ('♢', '3'),
 ('♣', '3'),
 ('♠', '4'),
 ('♡', '4'),
 ('♢', '4'),
 ('♣', '4'),
 ('♠', '5'),
 ('♡', '5'),
 ('♢', '5'),
 ('♣', '5'),
 ('♠', '6'),
 ('♡', '6'),
 ('♢', '6'),
 ('♣', '6'),
 ('♠', '7'),
 ('♡', '7'),
 ('♢', '7'),
 ('♣', '7'),
 ('♠', '8'),
 ('♡', '8'),
 ('♢', '8'),
 ('♣', '8'),
 ('♠', '9'),
 ('♡', '9'),
 ('♢', '9'),
 ('♣', '9'),
 ('♠', '10'),
 ('♡', '10'),
 ('♢', '10'),
 ('♣', '10'),
 ('♠', 'J'),
 ('♡', 'J'),
 ('♢', 'J'),
 ('♣', 'J'),
 ('♠', 'Q'),
 ('♡', 'Q'),
 ('♢', 'Q'),
 ('♣', 'Q'),
 ('♠', 'K'),
 ('♡', 'K'),
 ('♢', 'K'),
 ('♣', 'K'),
 ('♠', 'A'),
 ('♡', 'A'),
 ('♢', 'A'),
 ('♣', 'A')]

W wielu przypadkach funkcje będą oczekiwać pewnego rodzaju sekwencji i nie będzie miało dla nich znaczenia, czy jest to lista, czy krotka. W takich sytuacjach przy opisywaniu argumentu funkcji warto użyć typu `typing.Sequence`:

In [23]:
from typing import List, Sequence

def square(elems: Sequence[float]) -> List[float]:
    return [x**2 for x in elems]

Użycie typu `Sequence` jest przykładem wykorzystania *duck typingu*. `Sequence` to każdy obiekt, który obsługuje funkcję `len()` oraz metodę `.__getitem__()`, niezależnie od swojego faktycznego typu.

### Aliasy typów (*Type Aliases*)

Adnotacje typów mogą stać się dość nieczytelne podczas pracy z typami zagnieżdżonymi, takimi jak nasza talia kart. Trzeba dłuższą chwilę przyglądać się zapisowi `List[Tuple[str, str]]`, zanim zorientujemy się, że odpowiada on właśnie naszej reprezentacji talii.

Zastanówmy się teraz, jak należałoby opisać za pomocą adnotacji funkcję `deal_hands()`:

In [ ]:
def deal_hands(
    deck: List[Tuple[str, str]]
) -> Tuple[
    List[Tuple[str, str]],
    List[Tuple[str, str]],
    List[Tuple[str, str]],
    List[Tuple[str, str]],
]:
    """Deal the cards in the deck into four hands"""
    return (deck[0::4], deck[1::4], deck[2::4], deck[3::4])

Pamiętajmy, że adnotacje typów są zwykłymi wyrażeniami języka Python. Oznacza to, że możemy definiować własne aliasy typów, przypisując je do nowych zmiennych. Możemy na przykład utworzyć aliasy typów `Card` oraz `Deck`:

In [2]:
from typing import List, Tuple

Card = Tuple[str, str]
Deck = List[Card]

Typu `Card` można teraz używać w adnotacjach typów lub przy definiowaniu nowych aliasów, tak jak w przypadku `Deck` w powyższym przykładzie. Dzięki wykorzystaniu tych aliasów adnotacje funkcji `deal_hands()` stają się znacznie bardziej czytelne:

In [3]:
def deal_hands(deck: Deck) -> Tuple[Deck, Deck, Deck, Deck]:
    """Deal the cards in the deck into four hands"""
    return (deck[0::4], deck[1::4], deck[2::4], deck[3::4])

Aliasy typów świetnie sprawdzają się w zwiększaniu czytelności kodu oraz jasnym wyrażaniu jego intencji. Jednocześnie można je w każdej chwili podejrzeć (inspekcjonować), aby sprawdzić, co dokładnie reprezentują:

In [ ]:
from typing import List, Tuple
Card = Tuple[str, str]
Deck = List[Card]

Deck
typing.List[typing.Tuple[str, str]]

Zwróćmy uwagę, że po wypisaniu zmiennej `Deck` (za pomocą funkcji `print()`) wyświetlona informacja pokazuje, iż jest ona aliasem dla listy dwuelementowych krotek przechowujących napisy.